In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# **Mlflow and dagshub initialisation**

In [5]:
!pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 634.8 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 2.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 34.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [7]:
import dagshub
import mlflow

dagshub.init(repo_owner="luchit22", repo_name="ML-fraud-detection",mlflow=True)
mlflow.set_experiment("logistic_regression")

Initialized MLflow to track repo "luchit22/ML-fraud-detection"

Repository luchit22/ML-fraud-detection initialized!

<Experiment: artifact_location='mlflow-artifacts:/8fd1e66c18d241dcb3bccc96c04e5184', creation_time=1777630865098, experiment_id='0', last_update_time=1777630865098, lifecycle_stage='active', name='logistic_regression', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# **data cleaning**

In [10]:
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")

train_transaction.columns = train_transaction.columns.str.replace("-", "_")
train_identity.columns = train_identity.columns.str.replace("-", "_")

train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print(train.shape)

(590540, 434)


In [11]:
y = train["isFraud"]
X = train.drop(columns=["isFraud"])

In [13]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [14]:
def add_features(df):
    df = df.copy()
    
    df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])
    df["TransactionDT_days"] = df["TransactionDT"] / (3600 * 24)
    df["TransactionDT_hours"] = df["TransactionDT"] / 3600
    df["num_missing"] = df.isnull().sum(axis=1)
    
    return df

In [15]:
X_train = add_features(X_train)
X_valid = add_features(X_valid)

In [16]:
missing_ratio = X_train.isnull().mean()

cols_to_drop = missing_ratio[missing_ratio > 0.90].index.tolist()

X_train = X_train.drop(columns=cols_to_drop)
X_valid = X_valid.drop(columns=cols_to_drop)

print("Dropped columns:", len(cols_to_drop))

Dropped columns: 12


# **feature engineering**

In [17]:
X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train,
    y_train,
    train_size=80000,
    stratify=y_train,
    random_state=42
)

X_valid_sample, _, y_valid_sample, _ = train_test_split(
    X_valid,
    y_valid,
    train_size=30000,
    stratify=y_valid,
    random_state=42
)

In [21]:
class WOEEncoder:
    def __init__(self, min_samples=100):
        self.min_samples = min_samples
        self.woe_maps = {}
        self.default_woe = {}

    def fit(self, X, y):
        y = pd.Series(y).reset_index(drop=True)
        X = X.reset_index(drop=True)

        for col in X.columns:
            temp = pd.DataFrame({
                "feature": X[col].astype(str).fillna("missing"),
                "target": y
            })

            stats = temp.groupby("feature")["target"].agg(["count", "sum"])
            stats.columns = ["total", "bad"]
            stats["good"] = stats["total"] - stats["bad"]

            stats = stats[stats["total"] >= self.min_samples]

            if len(stats) == 0:
                self.woe_maps[col] = {}
                self.default_woe[col] = 0
                continue

            total_good = stats["good"].sum()
            total_bad = stats["bad"].sum()

            stats["good_dist"] = (stats["good"] + 0.5) / (total_good + 0.5)
            stats["bad_dist"] = (stats["bad"] + 0.5) / (total_bad + 0.5)

            stats["woe"] = np.log(stats["good_dist"] / stats["bad_dist"])

            self.woe_maps[col] = stats["woe"].to_dict()
            self.default_woe[col] = 0

        return self

    def transform(self, X):
        X = X.reset_index(drop=True)
    
        data = {}
    
        for col in X.columns:
            values = X[col].astype(str).fillna("missing")
            mapped = values.map(self.woe_maps.get(col, {})).fillna(0)
            data[col] = mapped
    
        X_new = pd.DataFrame(data, index=X.index)
    
        return X_new

In [22]:
woe_encoder = WOEEncoder(min_samples=100)

woe_encoder.fit(X_train_sample, y_train_sample)

X_train_woe = woe_encoder.transform(X_train_sample)
X_valid_woe = woe_encoder.transform(X_valid_sample)

print(X_train_woe.shape)
print(X_valid_woe.shape)

(80000, 425)
(30000, 425)


# **feature selection**

In [23]:
def calculate_iv(X_woe, y):
    iv_values = {}

    y = pd.Series(y).reset_index(drop=True)
    X_woe = X_woe.reset_index(drop=True)

    for col in X_woe.columns:
        temp = pd.DataFrame({
            "woe": X_woe[col],
            "target": y
        })

        grouped = temp.groupby("woe")["target"].agg(["count", "sum"])
        grouped.columns = ["total", "bad"]
        grouped["good"] = grouped["total"] - grouped["bad"]

        total_good = grouped["good"].sum()
        total_bad = grouped["bad"].sum()

        grouped["good_dist"] = (grouped["good"] + 0.5) / (total_good + 0.5)
        grouped["bad_dist"] = (grouped["bad"] + 0.5) / (total_bad + 0.5)

        grouped["iv"] = (
            grouped["good_dist"] - grouped["bad_dist"]
        ) * np.log(grouped["good_dist"] / grouped["bad_dist"])

        iv_values[col] = grouped["iv"].sum()

    return pd.Series(iv_values).sort_values(ascending=False)

In [24]:
iv_scores = calculate_iv(X_train_woe, y_train_sample)

iv_scores.head(30)

V201                  1.036937
V258                  1.006154
V200                  0.990983
V257                  0.977578
V189                  0.936643
V246                  0.933454
V244                  0.900064
V188                  0.892432
V243                  0.876617
V259                  0.872910
V199                  0.872210
V242                  0.864620
V190                  0.839719
V187                  0.828250
num_missing           0.805059
V245                  0.802693
C4                    0.799643
V186                  0.776674
C8                    0.760528
C12                   0.758584
C7                    0.728827
V232                  0.713475
V222                  0.710662
V233                  0.701699
V219                  0.700502
V218                  0.694088
V229                  0.687974
TransactionAmt        0.686575
TransactionAmt_log    0.686575
V171                  0.685536
dtype: float64

In [25]:
selected_features_iv = iv_scores[
    (iv_scores > 0.02) & (iv_scores < 1.0)
].index.tolist()

print("Selected by IV:", len(selected_features_iv))

Selected by IV: 381


In [26]:
X_train_iv = X_train_woe[selected_features_iv]
X_valid_iv = X_valid_woe[selected_features_iv]

In [27]:
corr_matrix = X_train_iv.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_features = [
    col for col in upper.columns
    if any(upper[col] > 0.90)
]

X_train_selected = X_train_iv.drop(columns=high_corr_features)
X_valid_selected = X_valid_iv.drop(columns=high_corr_features)

print("Removed by correlation:", len(high_corr_features))
print("Final features:", X_train_selected.shape[1])

Removed by correlation: 189
Final features: 192


In [28]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_selected)
X_valid_scaled = scaler.transform(X_valid_selected)

In [29]:
param_grid = [
    {"C": 0.01, "solver": "liblinear", "penalty": "l1"},
    {"C": 0.01, "solver": "liblinear", "penalty": "l2"},
    {"C": 0.1,  "solver": "liblinear", "penalty": "l1"},
    {"C": 0.1,  "solver": "liblinear", "penalty": "l2"},
    {"C": 1.0,  "solver": "liblinear", "penalty": "l1"},
    {"C": 1.0,  "solver": "liblinear", "penalty": "l2"},
    {"C": 10.0, "solver": "liblinear", "penalty": "l1"},
    {"C": 10.0, "solver": "liblinear", "penalty": "l2"},

    {"C": 0.01, "solver": "lbfgs", "penalty": "l2"},
    {"C": 0.1,  "solver": "lbfgs", "penalty": "l2"},
    {"C": 1.0,  "solver": "lbfgs", "penalty": "l2"},
    {"C": 10.0, "solver": "lbfgs", "penalty": "l2"},

    {"C": 0.01, "solver": "saga", "penalty": "l1"},
    {"C": 0.01, "solver": "saga", "penalty": "l2"},
    {"C": 0.1,  "solver": "saga", "penalty": "l1"},
    {"C": 0.1,  "solver": "saga", "penalty": "l2"},
    {"C": 1.0,  "solver": "saga", "penalty": "l1"},
    {"C": 1.0,  "solver": "saga", "penalty": "l2"},
    {"C": 10.0, "solver": "saga", "penalty": "l1"},
    {"C": 10.0, "solver": "saga", "penalty": "l2"},
]

In [30]:
results = []
best_model = None
best_valid_auc = 0
best_params = None

for params in param_grid:
    run_name = f"LR_C_{params['C']}_solver_{params['solver']}_penalty_{params['penalty']}"

    with mlflow.start_run(run_name=run_name):
        model = LogisticRegression(
            C=params["C"],
            solver=params["solver"],
            penalty=params["penalty"],
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )

        model.fit(X_train_scaled, y_train_sample)

        train_proba = model.predict_proba(X_train_scaled)[:, 1]
        valid_proba = model.predict_proba(X_valid_scaled)[:, 1]

        train_auc = roc_auc_score(y_train_sample, train_proba)
        valid_auc = roc_auc_score(y_valid_sample, valid_proba)
        auc_gap = train_auc - valid_auc

        mlflow.log_params(params)
        mlflow.log_param("model", "LogisticRegression")
        mlflow.log_param("encoding", "WOE")
        mlflow.log_param("feature_selection_1", "IV")
        mlflow.log_param("feature_selection_2", "correlation_filter")
        mlflow.log_param("iv_min_threshold", 0.02)
        mlflow.log_param("iv_max_threshold", 1.0)
        mlflow.log_param("correlation_threshold", 0.90)
        mlflow.log_param("class_weight", "balanced")
        mlflow.log_param("dropped_missing_columns", len(cols_to_drop))
        mlflow.log_param("removed_high_corr_features", len(high_corr_features))
        mlflow.log_param("num_final_features", X_train_selected.shape[1])

        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("valid_auc", valid_auc)
        mlflow.log_metric("auc_gap", auc_gap)

        results.append({
            **params,
            "train_auc": train_auc,
            "valid_auc": valid_auc,
            "auc_gap": auc_gap
        })

        if valid_auc > best_valid_auc:
            best_valid_auc = valid_auc
            best_model = model
            best_params = params

results_df = pd.DataFrame(results).sort_values("valid_auc", ascending=False)
results_df

🏃 View run LR_C_0.01_solver_liblinear_penalty_l1 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0/runs/1ccc690ee7784357a919070e492008a7
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0
🏃 View run LR_C_0.01_solver_liblinear_penalty_l2 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0/runs/ae96aab9898540a79d3423caa4999f62
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0
🏃 View run LR_C_0.1_solver_liblinear_penalty_l1 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0/runs/10ae3730e9d749748f34b9ea57173534
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0
🏃 View run LR_C_0.1_solver_liblinear_penalty_l2 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0/runs/102ee99586f84eb4829929cdd8c6a1af
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflo

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


🏃 View run LR_C_0.1_solver_saga_penalty_l1 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0/runs/146f84c8f5564943889d014fad8b22b4
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


🏃 View run LR_C_0.1_solver_saga_penalty_l2 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0/runs/4800cdb61d274ab4ada0936d40932b53
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0


...

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


🏃 View run LR_C_10.0_solver_saga_penalty_l1 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0/runs/43003798efbd4c9482b890287806afb5
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


🏃 View run LR_C_10.0_solver_saga_penalty_l2 at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0/runs/82e396723cee495fb1260e269098ecb9
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0


,C,solver,penalty,train_auc,valid_auc,auc_gap
0,0.01,liblinear,l1,0.885833,0.865671,0.020162
12,0.01,saga,l1,0.885854,0.865635,0.020219
15,0.10,saga,l2,0.886960,0.865360,0.021600
1,0.01,liblinear,l2,0.887547,0.865107,0.022439
13,0.01,saga,l2,0.887563,0.865069,0.022494
8,0.01,lbfgs,l2,0.887563,0.865042,0.022521
2,0.10,liblinear,l1,0.887564,0.864963,0.022600
3,0.10,liblinear,l2,0.887610,0.864829,0.022781
9,0.10,lbfgs,l2,0.887616,0.864828,0.022788
4,1.00,liblinear,l1,0.887611,0.864814,0.022797


In [31]:
with mlflow.start_run(run_name="LogisticRegression_Best_WOE_IV_Correlation"):
    mlflow.log_params(best_params)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("encoding", "WOE")
    mlflow.log_param("feature_selection", "IV + correlation_filter")
    mlflow.log_param("scaling", "StandardScaler")
    mlflow.log_param("num_final_features", X_train_selected.shape[1])

    mlflow.log_metric("best_valid_auc", best_valid_auc)

    mlflow.sklearn.log_model(best_model, artifact_path="logistic_regression_model")
    mlflow.sklearn.log_model(scaler, artifact_path="scaler")

print("Best params:", best_params)
print("Best validation AUC:", best_valid_auc)

2026/05/01 15:14:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 15:14:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/01 15:14:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 15:14:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

🏃 View run LogisticRegression_Best_WOE_IV_Correlation at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0/runs/270aa104bdf84a6d88cec4542bf30d08
🧪 View experiment at: https://dagshub.com/luchit22/ML-fraud-detection.mlflow/#/experiments/0
Best params: {'C': 0.01, 'solver': 'liblinear', 'penalty': 'l1'}
Best validation AUC: 0.8656707459495024
